In [18]:
import os
from os.path import join as pjoin
from shutil import copy2, copytree
import datetime
import re


In [ ]:
# today = datetime.date.today()
# apps_dir="repair-benchmark/apps"
# for app in os.listdir(apps_dir):
#     for agent in ["claude","codex", "qwen"]:
#         generated_paths=[pjoin(apps_dir,app,"issues",pr_,"generated",agent) for pr_ in os.listdir(pjoin(apps_dir,app,"issues")) if not pr_.endswith("_private")]
#         for path in generated_paths:
#             generated_files_folders=[pjoin(path, f) for f in os.listdir(path)]
#             for file_folder in generated_files_folders:
#                 # print(file_folder)
#                 if file_folder.endswith(".patch"):
#                     #the name should be like generated_{today's date in the format of ddMMMyyyy}/{agent}/{pr_number}/{file_name}
#                     #date should be collected from the current date using an import
#                     copy2(file_folder, pjoin("generated_" + today.strftime("%d%b%Y"),agent,file_folder.split("/")[-4],file_folder.split("/")[-1]))
#                 elif os.path.isdir(file_folder):
#                     copytree(file_folder, pjoin("generated_" + today.strftime("%d%b%Y"),agent,file_folder.split("/")[-4],file_folder.split("/")[-1]), dirs_exist_ok=True)
#                 else:
#                     print(f"These are not supposed to be here: {file_folder}")


In [ ]:

# apps_dir = "repair-benchmark/apps"
# today = datetime.date.today()  # still used only for naming the destination folder

# attempt_pattern = re.compile(r"attempt_(\d{8}_\d{6})")

# for app in os.listdir(apps_dir):
#     for agent in ["claude", "codex", "qwen"]:
#         generated_paths = [
#             pjoin(apps_dir, app, "issues", pr_, "generated", agent)
#             for pr_ in os.listdir(pjoin(apps_dir, app, "issues"))
#             if not pr_.endswith("_private")
#         ]
#         for path in generated_paths:
#             if not os.path.isdir(path):
#                 continue

#             generated_files_folders = [pjoin(path, f) for f in os.listdir(path)]

#             # Group entries by their attempt timestamp, keep track of the latest
#             latest_timestamp = None
#             latest_entries = []

#             for file_folder in generated_files_folders:
#                 basename = os.path.basename(file_folder)
#                 match = attempt_pattern.search(basename)
#                 if not match:
#                     print(f"These are not supposed to be here: {file_folder}")
#                     continue

#                 timestamp = match.group(1)  # e.g. "20260909_183845"

#                 if latest_timestamp is None or timestamp > latest_timestamp:
#                     latest_timestamp = timestamp
#                     latest_entries = [file_folder]
#                 elif timestamp == latest_timestamp:
#                     latest_entries.append(file_folder)

#             # Copy only the entries belonging to the latest attempt
#             for file_folder in latest_entries:
#                 if file_folder.endswith(".patch"):
#                     copy2(file_folder, pjoin("generated_" + today.strftime("%d%b%Y"), agent, file_folder.split("/")[-4], file_folder.split("/")[-1]))
#                 elif os.path.isdir(file_folder):
#                     copytree(file_folder, pjoin("generated_" + today.strftime("%d%b%Y"), agent, file_folder.split("/")[-4], file_folder.split("/")[-1]), dirs_exist_ok=True)

In [ ]:
#Instead of going into all the apps and issues, we can parse the run_all.txt file to get the (pr_path, agent) pairs.
# This will ensure we copy the output of the commands we just ran, and not any old output that might be lying around from previous runs.
apps_dir = "repair-benchmark/apps"
today = datetime.date.today()  # still used only for naming the destination folder

attempt_pattern = re.compile(r"attempt_(\d{8}_\d{6})")
cmd_pattern = re.compile(r"--pr\s+(\S+)\s+--model\s+(\S+)")

# Parse run_all.txt to get (pr_path, agent) pairs instead of walking the whole apps_dir tree
with open("run_all.txt", "r") as f:
    pr_agent_pairs = set()
    for line in f:
        match = cmd_pattern.search(line)
        if match:
            pr_path, agent = match.group(1), match.group(2)
            pr_agent_pairs.add((pr_path, agent))

# Only process codex entries (change/remove this filter if you want all agents)
pr_agent_pairs = [(pr, agent) for pr, agent in pr_agent_pairs if agent in ["claude","codex","qwen"]]

for pr_path, agent in pr_agent_pairs:
    path = pjoin(pr_path, "generated", agent)

    if not os.path.isdir(path):
        continue

    generated_files_folders = [pjoin(path, f) for f in os.listdir(path)]

    # Group entries by their attempt timestamp, keep track of the latest
    latest_timestamp = None
    latest_entries = []

    for file_folder in generated_files_folders:
        basename = os.path.basename(file_folder)
        match = attempt_pattern.search(basename)
        if not match:
            print(f"These are not supposed to be here: {file_folder}")
            continue

        timestamp = match.group(1)  # e.g. "20260909_183845"

        if latest_timestamp is None or timestamp > latest_timestamp:
            latest_timestamp = timestamp
            latest_entries = [file_folder]
        elif timestamp == latest_timestamp:
            latest_entries.append(file_folder)

    # Copy only the entries belonging to the latest attempt
    for file_folder in latest_entries:
        if file_folder.endswith(".patch"):
            os.makedirs(pjoin("generated_" + today.strftime("%d%b%Y"), agent, file_folder.split("/")[-4]), exist_ok=True)
            copy2(file_folder, pjoin("generated_" + today.strftime("%d%b%Y"), agent, file_folder.split("/")[-4], file_folder.split("/")[-1]))
        #elif os.path.isdir(file_folder):
            #copytree(file_folder, pjoin("generated_" + today.strftime("%d%b%Y"), agent, file_folder.split("/")[-4], file_folder.split("/")[-1]), dirs_exist_ok=True)